# MLP Two Digit Addition

## Imports

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

## Creates the Dataset

In [3]:
rows = []

for a in range(10):
    for b in range(10):
        for c in range(10):
            x = torch.zeros(30)

            # one-hot encode the three digits
            x[a] = 1.0
            x[10 + b] = 1.0
            x[20 + c] = 1.0

            total = a + b + c

            # ones digit and carry (tens digit)
            rows.append((x, total % 10, total // 10))

X3 = torch.stack([r[0] for r in rows])
Y_ones = torch.tensor([r[1] for r in rows])
Y_carry = torch.tensor([r[2] for r in rows])

print('Dataset created')
print('X3 shape:', X3.shape)
print('Y_ones shape:', Y_ones.shape)
print('Y_carry shape:', Y_carry.shape)

Dataset created
X3 shape: torch.Size([1000, 30])
Y_ones shape: torch.Size([1000])
Y_carry shape: torch.Size([1000])


## Architecture

input (32) -> hidden layer 1 (64) (frozen in Model C) -> hidden layer 2 (64) -> ones head (10 classes) and carry head (2 classes)

In [4]:
class MultiDigitMLP(nn.Module):
    """Processes one digit column: one-hot(a) + one-hot(b) + carry_in."""
    def __init__(self, freeze_layer1=False):
        super().__init__()
        self.hidden1    = nn.Linear(30, 64)
        self.hidden2    = nn.Linear(64, 64)
        self.ones_head  = nn.Linear(64, 10)
        self.carry_head = nn.Linear(64,  3)

        #locks hidden1 for Model C
        if freeze_layer1:
            for p in self.hidden1.parameters():
                p.requires_grad_(False)

    def forward(self, x):
        h1 = self.hidden1(x).clamp(min=0)   
        h2 = self.hidden2(h1).clamp(min=0)
        return self.ones_head(h2), self.carry_head(h2)

def accuracy(logits, targets):
    return (logits.argmax(1) == targets).sum().item()

print('Architecture defined')

Architecture defined


## Helper
trains the model and takes the heads you want to be part of the training as input parameters

In [5]:
def train(model, heads='both', max_epochs=50_000, lr=1e-3):
    params = [p for p in model.parameters() if p.requires_grad]
    opt    = optim.Adam(params, lr=lr)
    ce     = nn.CrossEntropyLoss()
    N      = len(X3)

    t0 = time.time()
    for epoch in range(1, max_epochs + 1):
        opt.zero_grad()
        lo, lc = model(X3)
        loss = ce(lo, Y_ones)
        if heads == 'both':
            loss = loss + ce(lc, Y_carry)
        loss.backward()
        opt.step()

        co = accuracy(lo, Y_ones)
        cc = accuracy(lc, Y_carry)
        if epoch % 5000 == 0:
            print(f'  epoch {epoch:5d}  loss={loss.item():.4f}  ones={co}/{N}  carry={cc}/{N}')
        done = (co == N) if heads == 'ones' else (co == N and cc == N)
        if done:
            print(f'  100% at epoch {epoch}')
            break
    return time.time() - t0

print('Training helper defined')

Training helper defined


## Model A — ones head only

In [6]:
model_a = MultiDigitMLP()
print('Training Model A (ones only)...')
t_a = train(model_a, heads='ones')
print(f'Model A training time: {t_a:.2f}s')

model_a.eval()
for p in model_a.parameters():
    p.requires_grad_(False)

Training Model A (ones only)...
  100% at epoch 405
Model A training time: 0.74s


## Verify Model A

In [7]:
with torch.no_grad():
    lo, lc = model_a(X3)
    print(f'Model A  ones accuracy : {accuracy(lo, Y_ones)}/1000')
    print(f'Model A  carry accuracy: {accuracy(lc, Y_carry)}/1000  (untrained)')

Model A  ones accuracy : 1000/1000
Model A  carry accuracy: 301/1000  (untrained)


## Model B — both heads from scratch

In [8]:
model_b = MultiDigitMLP()
print('Training Model B (both heads from scratch)...')
t_b = train(model_b, heads='both')
print(f'Model B training time: {t_b:.2f}s')

model_b.eval()
for p in model_b.parameters():
    p.requires_grad_(False)

Training Model B (both heads from scratch)...
  100% at epoch 678
Model B training time: 1.74s


## Verify Model B

In [9]:
with torch.no_grad():
    lo, lc = model_b(X3)
    print(f'Model B  ones accuracy : {accuracy(lo, Y_ones)}/1000')
    print(f'Model B  carry accuracy: {accuracy(lc, Y_carry)}/1000')

Model B  ones accuracy : 1000/1000
Model B  carry accuracy: 1000/1000


## Model C — fine-tune from Model A with first layer frozen

load_state_dict copies all of Model A's weights. Only hidden2, ones_head, and carry_head receive gradients.

In [10]:
model_c = MultiDigitMLP(freeze_layer1=True)
model_c.load_state_dict(model_a.state_dict())   # warm-start from Model A

# Ensure everything except h1 is trainable
for name, p in model_c.named_parameters():
    if 'h1' not in name:
        p.requires_grad_(True)

print('Model C: h1 frozen, all other layers trainable')
t_c = train(model_c, heads='both')
print(f'Model C training time: {t_c:.2f}s')
model_c.eval()

Model C: h1 frozen, all other layers trainable
  100% at epoch 361
Model C training time: 0.85s


MultiDigitMLP(
  (hidden1): Linear(in_features=30, out_features=64, bias=True)
  (hidden2): Linear(in_features=64, out_features=64, bias=True)
  (ones_head): Linear(in_features=64, out_features=10, bias=True)
  (carry_head): Linear(in_features=64, out_features=3, bias=True)
)

## Verify Model C

In [11]:
with torch.no_grad():
    lo, lc = model_c(X3)
    print(f'Model C  ones accuracy : {accuracy(lo, Y_ones)}/1000')
    print(f'Model C  carry accuracy: {accuracy(lc, Y_carry)}/1000')

Model C  ones accuracy : 1000/1000
Model C  carry accuracy: 1000/1000


## Stack for 2-digit addition

Two calls to a MultiDigitMLP, with the carry output of the ones column feeding into the tens column.

In [12]:
def add_3digits(net, a, b, c):
    x = torch.zeros(30)

    x[a] = 1.0
    x[10 + b] = 1.0
    x[20 + c] = 1.0

    with torch.no_grad():
        lo, lc = net(x.unsqueeze(0))

    ones = lo.argmax(1).item()
    carry = lc.argmax(1).item()

    return 10 * carry + ones

def exhaustive_check(net, name):
    errors = []

    for a in range(10):
        for b in range(10):
            for c in range(10):
                pred = add_3digits(net, a, b, c)

                if pred != a + b + c:
                    errors.append((a, b, c, pred, a + b + c))

    correct = 1000 - len(errors)
    print(f'{name}: {correct}/1000 = {correct/1000:.2%}')

    return correct / 1000

print('Stack helper defined')

Stack helper defined


## Exhaustive verification — all 10,000 pairs (0+0 to 99+99)

In [13]:
score_a = exhaustive_check(model_a, 'Model A (ones-only train):') # should be about 25% accuracy for 2 digits because carry is random
score_b = exhaustive_check(model_b, 'Model B (both heads, scratch):')
score_c = exhaustive_check(model_c, 'Model C (frozen h1, fine-tune):')

Model A (ones-only train):: 301/1000 = 30.10%
Model B (both heads, scratch):: 1000/1000 = 100.00%
Model C (frozen h1, fine-tune):: 1000/1000 = 100.00%


## Compare training times

In [14]:
print('=' * 50)
print(f'Model A  (ones only, from scratch)   : {t_a:.2f}s   2-digit acc: {score_a}/10000')
print(f'Model B  (both heads, from scratch)  : {t_b:.2f}s   2-digit acc: {score_b}/10000')
print(f'Model C  (both heads, frozen h1)     : {t_c:.2f}s   2-digit acc: {score_c}/10000')
print('=' * 50)

if t_c < t_b:
    print(f'C was faster than B by {t_b - t_c:.2f}s')
else:
    print(f'B was faster than C by {t_c - t_b:.2f}s')

if t_a + t_c < t_b:
    print(f'A+C together were faster than B by {t_b - (t_a + t_c):.2f}s')
else:
    print(f'B was faster than A+C together by {(t_a + t_c) - t_b:.2f}s')

Model A  (ones only, from scratch)   : 0.74s   2-digit acc: 0.301/10000
Model B  (both heads, from scratch)  : 1.74s   2-digit acc: 1.0/10000
Model C  (both heads, frozen h1)     : 0.85s   2-digit acc: 1.0/10000
C was faster than B by 0.89s
A+C together were faster than B by 0.15s
